In [1]:
from dotenv import load_dotenv
from huggingface_hub import login
import torch
from datasets import load_dataset
from tqdm import tqdm
import os

# Silence warnings and clean up output
import transformers
from transformers import pipeline
transformers.logging.set_verbosity_error() 
transformers.utils.logging.disable_progress_bar()
import warnings, logging

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars() 

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cpu


In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
import numpy as np
from numpy.linalg import norm

pairs = [
        # right: same bird, different wording
        ("red-winged blackbird", "red winged blackbird", "right"),
        ("This is a Kentucky Warbler", "kentucky warbler", "right"),
        ("Anna's Hummingbird", "anna hummingbird", "right"),
        ("Northern Cardinal", "cardinal", "right"),
        ("Baltimore oriole.", "baltimore oriole", "right"),
        ("The bird is an Indigo Bunting", "indigo bunting", "right"),
        ("pied-billed grebe", "pied billed grebe", "right"),

        # close: wrong species in the same family, often a one-word difference
        ("rusty blackbird", "red winged blackbird", "close"),
        ("brewer blackbird", "rusty blackbird", "close"),
        ("california gull", "western gull", "close"),
        ("herring gull", "california gull", "close"),
        ("hooded warbler", "kentucky warbler", "close"),
        ("laysan albatross", "black footed albatross", "close"),
        ("fish crow", "american crow", "close"),
        ("orchard oriole", "baltimore oriole", "close"),
        ("lazuli bunting", "indigo bunting", "close"),
        ("red bellied woodpecker", "red headed woodpecker", "close"),
        ("summer tanager", "scarlet tanager", "close"),
        ("green jay", "blue jay", "close"),

        # vague: right family but not a species name
        ("blackbird", "red winged blackbird", "vague"),
        ("a gull", "western gull", "vague"),
        ("warbler", "kentucky warbler", "vague"),
        ("hummingbird", "anna hummingbird", "vague"),

        # wrong: unrelated birds
        ("blue jay", "black footed albatross", "wrong"),
        ("mallard", "ruby throated hummingbird", "wrong"),
        ("cardinal", "laysan albatross", "wrong"),
        ("common raven", "painted bunting", "wrong"),
]

for pair in pairs:
    embedding1 = model.encode(pair[0])
    embedding2 = model.encode(pair[1])

    similarity = np.dot(embedding1, embedding2) / (
        norm(embedding1) * norm(embedding2)
    )

    print(pair[2], similarity)



right 0.9873299
right 0.94018584
right 0.88429207
right 0.7676479
right 0.9865618
right 0.78226554
right 0.96182644
close 0.60933113
close 0.6339238
close 0.699423
close 0.6011231
close 0.68891215
close 0.6769653
close 0.7096864
close 0.56299865
close 0.5604811
close 0.9126701
close 0.7230569
close 0.8051142
vague 0.838584
vague 0.7487887
vague 0.8203127
vague 0.7388581
wrong 0.30260596
wrong 0.25590268
wrong 0.33608428
wrong 0.23166586


In [ ]:
# looks like right ranges from 0.75 - 1
# close ranges from 0.6 - 0.8
# vague ranges from 0.7 - 0.8
# wrong ranges from 0.2 - 0.3
# cosine similarity is probably not the best indicator of trust

In [ ]:
# Side not, AI recommends using batching to minimise number of calls to HF:
# Useful knowledge for future writing
sentences = [sentence for pair in pairs for sentence in pair[:2]]
embeddings = model.encode(sentences)

for i, pair in enumerate(pairs):
    embedding1 = embeddings[2 * i]
    embedding2 = embeddings[2 * i + 1]

    similarity = np.dot(embedding1, embedding2) / (
        norm(embedding1) * norm(embedding2)
    )

    print(f"{pair[2]:6} | {similarity:.4f} | {pair[0]} vs {pair[1]}")

In [ ]:
# check valid set
from llava_peft.data import get_dataset
class_names = get_dataset("birds").class_names
print(len(class_names))
in_class = []
for pair in pairs:
	if pair[1] in class_names: in_class.append(pair[0])

print(len(in_class), len(pairs))

200
27 27


In [ ]:
# Encode class names
class_emb = model.encode(class_names, normalize_embeddings=True)
print(class_emb.shape) # shape (200, 384)

# Embed the predictions and compare to the class, labels no longer required
preds = [pair[0] for pair in pairs]
pred_embs = model.encode(preds, normalize_embeddings=True)  # (27, 384)

for i, (pred, label, category) in enumerate(pairs):
        scores = class_emb @ pred_embs[i] # cosine score against every class

        # argsort is ascending, so reverse it to put the best match first
        order = np.argsort(scores)[::-1]
        top1 = class_names[order[0]]
        top5 = [class_names[j] for j in order[:5]]
        gap = scores[order[0]] - scores[order[1]]

        correct = top1 == label
        in_top5 = label in top5

        print(f"{category:6} | {'Y' if correct else 'N'} top1 | {'Y' if in_top5 else 'N'} top5 | gap {gap:.3f} | {pred!r} -> {top1!r} (label {label!r})")

(200, 384)
right  | Y top1 | Y top5 | gap 0.214 | 'red-winged blackbird' -> 'red winged blackbird' (label 'red winged blackbird')
right  | Y top1 | Y top5 | gap 0.148 | 'This is a Kentucky Warbler' -> 'kentucky warbler' (label 'kentucky warbler')
right  | Y top1 | Y top5 | gap 0.216 | "Anna's Hummingbird" -> 'anna hummingbird' (label 'anna hummingbird')
right  | Y top1 | Y top5 | gap 0.245 | 'Northern Cardinal' -> 'cardinal' (label 'cardinal')
right  | Y top1 | Y top5 | gap 0.366 | 'Baltimore oriole.' -> 'baltimore oriole' (label 'baltimore oriole')
right  | Y top1 | Y top5 | gap 0.179 | 'The bird is an Indigo Bunting' -> 'indigo bunting' (label 'indigo bunting')
right  | Y top1 | Y top5 | gap 0.261 | 'pied-billed grebe' -> 'pied billed grebe' (label 'pied billed grebe')
close  | N top1 | Y top5 | gap 0.366 | 'rusty blackbird' -> 'rusty blackbird' (label 'red winged blackbird')
close  | N top1 | Y top5 | gap 0.206 | 'brewer blackbird' -> 'brewer blackbird' (label 'rusty blackbird')
clo

In [ ]:
# Top 5 is inflated fro wrong answers, best metric here is to go with top 1
# 7/7 got top 1 -> the embeddings handle extra characters/ different wording with no problems